# gaitkeeper quickstart

Why does a walking policy that works in training fail in a second simulator? This notebook
runs [gaitkeeper](https://github.com/dundysm/gaitkeeper) on Unitree's own G1 policy and
reads the answer. CPU only, about three minutes. Run the cells in order.

In [ ]:
%env MUJOCO_GL=disable
%pip install -q "gaitkeeper[sim]"
!gaitkeeper --version

## 1. The demo: a policy that scores 0% on a waypoint tour

[unitree_rl_lab issue 145](https://github.com/unitreerobotics/unitree_rl_lab/issues/145)
reports the official G1 velocity policy scoring 0% in a third-party benchmark. Is the
benchmark's harness wrong, or the policy?

`demo` downloads that policy and the unitree_mujoco G1 scene (about 20 MB, pinned commits,
sha256 checked) and runs the setup end to end. Look for the `DEAD ZONE` lines.

In [ ]:
!gaitkeeper demo

The policy stands still for forward commands of 0.20 m/s or less and does not turn in
place. Most of the tour is small commands and in-place turns, so it fails with a harness
that follows the contract. That is a finding about the policy in this runner (evidence
level L1), not a verdict on anyone's harness.

## 2. `doctor`: the same check on any policy

`doctor` takes an exported policy, the config it was trained or deployed with, and a
MuJoCo scene. It reports the contract fields no file states, the commands the policy
ignores, and survival on a closed-loop waypoint tour.

In [ ]:
import os

D = os.path.expanduser("~/.cache/gaitkeeper")
!gaitkeeper doctor --onnx {D}/g1_rl_lab/policy.onnx --config {D}/g1_rl_lab/deploy.yaml --mjcf {D}/g1_unitree_mujoco/scene_29dof.xml --json report.json

Reading it:

1. **Contract**: 12 fields that `deploy.yaml` does not state, among them the simulation
   step and the decimation. A harness has to guess them. The results hold under the
   defaults listed.
2. **Command response**: the dead zones from the demo.
3. **Waypoint tour**: it completes the 90 s tour with its own arms, and lasts about 6 s
   once the harness moves the arms at random. The failure is the arm motion, not the walking.

The [walkthrough](https://github.com/dundysm/gaitkeeper/blob/main/docs/WALKTHROUGH.md)
explains each section. The report is also in `report.json`:

In [ ]:
import json

r = json.load(open("report.json"))
print("unknown contract fields:", len(r["unknown_fields"]))
for label, row in r["tour"].items():
    print(f"{label:22s} mean survival {row['mean_survival_s']:.1f} s")

## 3. Your own policy

Upload your files with the folder icon on the left, then:

```
!gaitkeeper doctor --onnx policy.onnx --config env.yaml --mjcf scene.xml
```

`--config` is read by content: an Isaac Lab `env.yaml` (add `--joint-order deploy.yaml`),
a Unitree `deploy.yaml`, a unitree_rl_gym or legged_gym config. See
[Run it on your policy](https://github.com/dundysm/gaitkeeper#run-it-on-your-policy).

For a cause and not only a finding, record a golden trace in your training simulator and
run `gaitkeeper verify`: [How it decides](https://github.com/dundysm/gaitkeeper#how-it-decides).